# 09 — finish the final-token causal repair (ONE notebook, full standard)

**Why.** The committed causal pipeline built the A-D direction from
`_pooled.npy` = *mean of the last 5 prompt tokens*. `analysis_plan.md` 4 fixes
the canonical direction on the **final** prompt token (`_final.npy`).
`cos(final, mean-last-5) ~ 0.78-0.87` at the ablation layers — a real
difference. This regenerates every causal result with `--pooling final_token`
so CF2 (confirmatory anchor), its WildGuard cross-check, the quadrant-C
McNemar, and the path-dependence results all match what the paper claims.

**No quality shortcuts.** Per DPO branch (M3, M3_direct, M3_alt, M3_direct_alt):

| run | rows | fills |
|---|---|---|
| held-out (plain) | ~1242 | CF2 **primary** (held-out 30 A) + **quadrant-C McNemar** + B/D descriptive |
| `--cross-fit 5` | 360 | cross_fitted n=120 + branch contrasts + 2x2 + circularity |
| `--all-ad-sensitivity` (`RUN_FULL_AD`) | 900 | `full_A_sensitivity` n=150 |

Then **both** judges — StrongREJECT (primary continuous) and WildGuard (the
preregistered independent binary cross-check, `analysis_plan.md` 10) — one 7B
model at a time. fp16, with a 4-bit fallback if fp16 OOMs on the T4.

**Resumable.** `v2_pipeline` skips a branch whose bound output exists; the
judge resumes from any prior final-token judged file. Cell 3 prints exactly
what is already done so a re-run after a dead session picks up where it stopped.

**Wall time from scratch:** ~4-5 h generation + ~3-4 h judging (both models) ~
**8-9 h**. Split it across two sessions if needed — generation and judging are
separate cells and each is independently resumable.

**Needs:** T4 runtime, Colab secret `HF_TOKEN` (with `google/gemma-2b` **and**
`allenai/wildguard` accepted), the `dpo_v2` Drive folder (654-row `_final.npy`).

## 1. Clone + branch tip + Drive + HF

In [ ]:
import os, sys, subprocess, glob, json
from pathlib import Path
from collections import Counter

REPO = "https://github.com/urosavurdic/dpo-safety-representations.git"
BR   = "agent/c-quadrant-end-to-end-e0e2317a"
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "origin", "--quiet"], check=True)
subprocess.run(["git", "checkout", "-B", BR, "origin/" + BR], check=True)  # branch TIP, not a stale pin
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", HEAD)

from google.colab import drive, userdata
drive.mount("/content/drive")
cands = (["/content/drive/MyDrive/dpo_v2"]
         + sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2"))
         + sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2")))
REAL = next((c for c in cands
             if glob.glob(os.path.join(c, "results", "activations", "*_final.npy"))), None)
assert REAL, "no dpo_v2 folder with activations found:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL
from src.colab_persist import bind, status_line
print(status_line(bind(persist_hf_cache=False)))   # results/ -> Drive symlink (auto-persist)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF ok")

RUN_FULL_AD = True   # set False to skip the n=150 full_A_sensitivity generation only

## 2. Env fix + fail-fast (before any GPU time)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as _plt
        _plt.is_torchao_available = lambda *a, **k: False
    except Exception:
        pass
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "no GPU - set the Colab runtime to a T4."

from src.training.model import load_stage_model
_m, _t = load_stage_model("M3")           # fail on a dep break HERE, not 20 min in
print("load_stage_model('M3') OK -", type(_m).__name__)
del _m, _t
import gc; gc.collect(); torch.cuda.empty_cache()

## 3. STATUS - what is done, what is left (no GPU)

In [ ]:
import numpy as np
BRANCHES = ["M3", "M3_direct", "M3_alt", "M3_direct_alt"]
BENCH = sorted(glob.glob("data/frozen_v2/benchmark_v2_*.jsonl"))[-1]

print("=== 654-row _final activations (needed to build the direction) ===")
ok = True
for b in BRANCHES:
    mp, fp = f"results/activations/{b}_metadata.json", f"results/activations/{b}_final.npy"
    if not (os.path.exists(mp) and os.path.exists(fp)):
        print(f"  {b:16s} MISSING"); ok = False; continue
    m = json.load(open(mp, encoding="utf-8", errors="replace"))
    n = np.load(fp, mmap_mode="r").shape[0]
    sp = sum(1 for r in m if r.get("split"))
    good = len(m) == 654 and n == 654 and sp == 300
    print(f"  {b:16s} meta={len(m)} npy={n} splits={sp}  {'OK' if good else '<-- STALE'}")
    ok = ok and good
assert ok, "activations not all 654-row - cannot build the final-token direction."

def nrows(p):
    try: return len(json.load(open(p, encoding="utf-8", errors="replace")))
    except Exception: return None

RUNS = [("held-out", "", None),           # plain: A/D/B/C  -> primary + quad-C McNemar
        ("xfit5",   "_xfit5", 360)]
if RUN_FULL_AD:
    RUNS.append(("fullAD", "_fullAD", 900))

print("\n=== final-token causal generations on Drive ===")
todo = []
for b in BRANCHES:
    for label, sfx, want in RUNS:
        p = f"results/raw/causal_ablation_v2_{b}_L24-28{sfx}_finaltoken.json"
        n = nrows(p)
        done = (n is not None) and (want is None and n > 300 or n == want)
        print(f"  {b:16s} {label:9s} {str(n):>6} rows  {'OK' if done else 'TODO'}")
        if not done:
            todo.append((b, label, sfx, want))

print("\n=== final-token judged files ===")
jfiles = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
RESUME = jfiles[-1] if jfiles else None
if RESUME:
    recs = json.load(open(RESUME)).get("records", [])
    st = lambda r: r.get("stage") or r.get("condition") or ""
    sr = lambda r: (r.get("strong_reject") or {}).get("judge_status") == "scored"
    wg = lambda r: (r.get("wildguard") or {}).get("judge_status") == "scored"
    ftA = [r for r in recs if "_ft_" in st(r) and r.get("quadrant") in (None, "A")]
    print(f"  newest: {os.path.basename(RESUME)}  ({len(recs)} recs)")
    print(f"  ft_ quadrant-A rows: {len(ftA)}  StrongREJECT={sum(map(sr,ftA))}  WildGuard={sum(map(wg,ftA))}")
    print(f"  judge_status: {json.load(open(RESUME)).get('judge_status')}")
else:
    print("  none yet")

print("\n" + "=" * 58 + "\nREMAINING WORK\n" + "=" * 58)
for b, label, *_ in todo: print(f"  GENERATE  {b:16s} {label}")
if not todo: print("  generation: COMPLETE")
print("  JUDGE     both models (cell 6) - resumes from the newest file above")
print("  POST      endpoints + quad-C McNemar (cell 8) - always re-run, cheap")
TODO = todo

## 4. GENERATE - held-out + cross-fit (+ full-A/D if RUN_FULL_AD)

Finished branches print `already bound ... skipping` in ~1 s. A best-effort
`git push` runs after each branch (Drive already holds everything - `results/`
is a symlink).

In [ ]:
def ckpt(label):
    try:
        subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
        subprocess.run(["git", "config", "user.name", "finaltoken (Colab)"], check=True)
        add = [f for f in glob.glob("results/raw/causal_ablation_v2_*finaltoken*") if os.path.isfile(f)]
        subprocess.run(["git", "add", "--"] + add, check=True, capture_output=True)
        if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
            subprocess.run(["git", "commit", "-m", f"final-token causal (Colab): {label}"],
                           check=True, capture_output=True)
            subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
            r = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
            print(f"    [ckpt {label}] push rc={r.returncode}"
                  + ("" if r.returncode == 0 else "  (Drive still holds it)"))
    except Exception as ex:
        print(f"    [ckpt {label}] git skipped ({ex}) - Drive holds it")

BASE = [sys.executable, "-m", "src.analysis.v2_pipeline", "causal", "--pooling", "final_token"]
for b in BRANCHES:
    for label, sfx, want in RUNS:
        cmd = BASE + ["--stage", b]
        if label == "xfit5":  cmd += ["--cross-fit", "5"]
        if label == "fullAD": cmd += ["--all-ad-sensitivity"]
        print(f"\n>>> {b} {label}"); sys.stdout.flush()
        r = subprocess.run(cmd)
        assert r.returncode == 0, f"generation FAILED: {b} {label}"
        p = f"results/raw/causal_ablation_v2_{b}_L24-28{sfx}_finaltoken.json"
        rows = json.load(open(p, encoding="utf-8", errors="replace"))
        conds = dict(Counter(x.get("stage") for x in rows))
        print(f"    {p}  {len(rows)} rows  {conds}")
        if want is not None:
            assert len(rows) == want, f"{b} {label}: expected {want}, got {len(rows)}"
        else:
            assert len(rows) > 300 and len(conds) == 3, f"{b} held-out: unexpected shape {conds}"
    ckpt(b)

ft = sorted(f for f in glob.glob("results/raw/causal_ablation_v2_*L24-28*finaltoken*.json")
            if not f.endswith("_binding.json"))
exp = len(BRANCHES) * len(RUNS)
print(f"\nfinal-token causal files: {len(ft)} (expected {exp})")
for f in ft: print("  ", f)
assert len(ft) == exp, f"expected {exp}, found {len(ft)}"
print("\nGENERATION COMPLETE.")

## 5. Judge probe - StrongREJECT + WildGuard, fp16 then 4-bit fallback

In [ ]:
import gc
from src.analysis.behavioral_judges import (LazyModelJudge, DEFAULT_STRONGREJECT_MODEL,
                                             DEFAULT_WILDGUARD_MODEL)

def probe(name, model_id, mode, four_bit):
    j = LazyModelJudge(name, model_id, load_4bit=four_bit, allow_download=True, mode=mode)
    ok = j.try_load()
    err = None if ok else j.load_error
    try: j.unload()
    except Exception: pass
    del j; gc.collect(); torch.cuda.empty_cache()
    return ok, err

LOAD_4BIT = False
for nm, mid, md_ in (("strong_reject", DEFAULT_STRONGREJECT_MODEL, "score_1_to_5"),
                     ("wildguard",     DEFAULT_WILDGUARD_MODEL,    "generate")):
    ok, err = probe(nm, mid, md_, four_bit=False)
    print(f"{nm:14s} fp16 : {'OK' if ok else 'FAIL - ' + str(err)[:160]}")
    if not ok:
        ok4, err4 = probe(nm, mid, md_, four_bit=True)
        print(f"{nm:14s} 4bit : {'OK' if ok4 else 'FAIL - ' + str(err4)[:160]}")
        assert ok4, (f"{nm} loads in neither fp16 nor 4-bit. fp16 err: {err}\n4-bit err: {err4}\n"
                     f"Usually the HF licence for {mid} is not accepted on this account.")
        LOAD_4BIT = True
print("\nLOAD_4BIT =", LOAD_4BIT, " (both judges will run)")

## 6. JUDGE - both models, one at a time, resume from any prior file

In [ ]:
Path("results/final_token_repair/manifests").mkdir(parents=True, exist_ok=True)
Path("results/final_token_repair/judges").mkdir(parents=True, exist_ok=True)

b0 = json.load(open(glob.glob("results/raw/causal_ablation_v2_*finaltoken*_binding.json")[0]))
mp = "results/final_token_repair/manifests/consolidated_finaltoken.json"
json.dump({"kind": "consolidated_response_manifest", "pooling": "final_token",
           "benchmark_sha256": b0.get("benchmark_sha256"),
           "split_manifest_sha256": b0.get("split_manifest_sha256"),
           "entries": [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")}
                       for f in ft]}, open(mp, "w"), indent=2)

prior = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
resume = prior[-1] if prior else None
print("resume-from:", os.path.basename(resume) if resume else "(none)")

cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", mp, "--run-live", "--allow-download",
       "--require-binding", "--reject-legacy",
       "--out-dir", "results/final_token_repair/judges"]
if not LOAD_4BIT: cmd += ["--no-4bit"]
if resume:        cmd += ["--resume-from", resume]
before = set(prior)
r = subprocess.run(cmd)     # streams progress live (per-100-row lines)
assert r.returncode == 0, "judge failed - read the traceback above"
new = sorted(set(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json")) - before)
jf = new[-1] if new else prior[-1]
print("\njudged file:", jf, round(os.path.getsize(jf) / 1e6, 1), "MB")
ckpt("judged")

## 7. Verify - every in-scope row scored by BOTH judges

In [ ]:
data = json.load(open(jf)); recs = data.get("records", [])
js = data.get("judge_status", {})
st = lambda r: r.get("stage") or r.get("condition") or ""
sr = lambda r: (r.get("strong_reject") or {}).get("judge_status") == "scored"
wg = lambda r: (r.get("wildguard") or {}).get("judge_status") == "scored"
xf  = [r for r in recs if "ft_xfit" in st(r)]
adA = [r for r in recs if "_ft_" in st(r) and "xfit" not in st(r) and r.get("quadrant") in (None, "A")]
print(f"ft_xfit      {len(xf):5d}  SR={sum(map(sr,xf)):5d}  WG={sum(map(wg,xf)):5d}")
print(f"ft_ held-out {len(adA):5d}  SR={sum(map(sr,adA)):5d}  WG={sum(map(wg,adA)):5d}")
print("judge_status:", js)
assert js.get("strong_reject") == "scored", "StrongREJECT did not complete"
assert js.get("wildguard") == "scored", ("WildGuard did not complete - it is the preregistered "
    "independent cross-check; do NOT proceed. Re-run cell 6.")
assert sum(map(sr, xf)) >= 1400 and sum(map(wg, xf)) >= 1400, "cross-fit not fully scored by both"
assert sum(map(sr, adA)) >= 300 and sum(map(wg, adA)) >= 300, "held-out A not fully scored by both"
print("\nOK - both judges complete.")

## 8. POST - final-token endpoints + quadrant-C McNemar + pooled-vs-final table

`--condition-infix ft_` matches `{stage}_ft_baseline` / `{stage}_ft_xfit_baseline`,
so only the final-token rows are read. CF1 is pooling-independent and is kept
from the pooled run.

In [ ]:
SUM = "results/final_token_repair/summaries"; Path(SUM).mkdir(parents=True, exist_ok=True)
EP = SUM + "/final_token_endpoints.json"
r = subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                    "--judged", jf, "--benchmark", BENCH,
                    "--condition-infix", "ft_", "--out", EP])
assert r.returncode == 0 and os.path.exists(EP), "POST failed"
e = json.load(open(EP)); pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))

def fmt(x):
    if not x or x.get("cf2") is None: return "n=0"
    return f"{x['cf2']:+.4f} [{x['ci_low']:+.4f},{x['ci_high']:+.4f}] n={x['n_effective_triples']}"

print("=" * 90)
print("FINAL-TOKEN CF2  vs  POOLED (mean-last-5)")
print("=" * 90)
comp = []
for b in BRANCHES:
    for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
        fb = e.get("CF2_by_stage", {}).get(b, {}).get(pop) or {}
        pb = pooled.get("CF2_by_stage", {}).get(b, {}).get(pop) or {}
        print(f"  {b:15s}{pop:20s} final {fmt(fb):38s}  pooled {fmt(pb)}")
        if fb.get("cf2") is not None and pb.get("cf2") is not None:
            comp.append({"stage": b, "population": pop, "final_token_cf2": fb["cf2"],
                         "pooled_cf2": pb["cf2"], "abs_diff": abs(fb["cf2"] - pb["cf2"]),
                         "final_ci": [fb["ci_low"], fb["ci_high"]],
                         "pooled_ci": [pb["ci_low"], pb["ci_high"]]})
    wgb = (e.get("CF2_by_stage", {}).get(b, {}).get("primary") or {}).get("secondary_binary_wildguard") or {}
    if wgb.get("point") is not None:
        print(f"  {b:15s}{'WildGuard(binary)':20s} final {wgb['point']:+.4f} "
              f"[{wgb['ci_low']:+.4f},{wgb['ci_high']:+.4f}] n={wgb['n_effective_triples']}")

xc = e.get("CF2_crossfit_branch_contrasts") or {}
f2 = xc.get("factorial_2x2") or {}
print("\ncross-fitted branch contrasts (final-token):")
for k, p in {**(xc.get("pairwise") or {}), **f2}.items():
    if p.get("estimate") is None: continue
    print(f"  {k:42s} {p['estimate']:+.4f} [{p['ci_low']:+.4f},{p['ci_high']:+.4f}]"
          + ("  <-- CI excl 0" if p.get("ci_excludes_zero") else ""))
assert (f2.get("corpus_x_history_interaction") or {}).get("estimate") is not None, \
    "2x2 interaction n/a - cross-fit not scored"

cb = (e.get("CF2_circularity_bias") or {}).get("per_branch") or {}
print("\ncircularity bias (est_split - cross_fitted, same rows, final-token):")
for stg, p in cb.items():
    print(f"  {stg:16s} {p['bias_estimation_minus_crossfit']:+.4f} [{p['ci_low']:+.4f},{p['ci_high']:+.4f}]")

# quadrant-C McNemar at final-token (regex; reads the held-out files)
print("\nquadrant-C McNemar (final-token, ablated_AD vs ablated_random, soft_deflection):")
MC = {}
for b in BRANCHES:
    hp = f"results/raw/causal_ablation_v2_{b}_L24-28_finaltoken.json"
    rr = subprocess.run([sys.executable, "-m", "src.analysis.mcnemar_causal_ablation",
                         "--file", hp, "--conditions", f"{b}_ft_ablated_AD", f"{b}_ft_ablated_random",
                         "--quadrant", "C", "--category", "soft_deflection"],
                        capture_output=True, text=True)
    print(f"  --- {b} ---")
    for ln in rr.stdout.splitlines():
        if any(k in ln for k in ("switched", "McNemar", "n=")): print("   ", ln.strip())
    MC[b] = rr.stdout

json.dump({"note": "final-token vs pooled (mean-last-5) CF2, all populations + WildGuard",
           "final_token_endpoints": EP, "rows": comp,
           "quadrant_c_mcnemar_final_token_stdout": MC},
          open(SUM + "/pooled_vs_final_token_CF2.json", "w"), indent=2)
print("\nwrote", SUM + "/pooled_vs_final_token_CF2.json")

try:
    subprocess.run(["git", "add", "--",
                    *glob.glob("results/raw/causal_ablation_v2_*finaltoken*"),
                    *glob.glob("results/final_token_repair/summaries/*.json"),
                    *glob.glob("results/final_token_repair/manifests/*.json")],
                   check=True, capture_output=True)
    if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
        subprocess.run(["git", "commit", "-m", "final-token causal repair: endpoints + McNemar (Colab)"],
                       check=True, capture_output=True)
        subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
        rr = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
        print("git push rc=", rr.returncode)
except Exception as ex:
    print("git push skipped:", ex, "- Drive holds everything under", REAL + "/results/")

## DONE - send back

Paste the **cell-8 output**, and (if the git push failed) download + send:

- `results/final_token_repair/summaries/final_token_endpoints.json`
- `results/final_token_repair/summaries/pooled_vs_final_token_CF2.json`

In [ ]:
try:
    from google.colab import files
    files.download("results/final_token_repair/summaries/final_token_endpoints.json")
    files.download("results/final_token_repair/summaries/pooled_vs_final_token_CF2.json")
except Exception as ex:
    print("download skipped:", ex, "- Drive UI:",
          REAL + "/results/final_token_repair/summaries/")